In [32]:
import torch
import torch.nn.functional as F
import clip
from PIL import Image
from pathlib import Path

In [18]:
images_folder = Path("images")
vectors = []

In [19]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

In [20]:
for image_path in images_folder.iterdir():
    preprocessed_image = preprocess(Image.open(image_path)).unsqueeze(0).to(device)

    with torch.no_grad():
        image_embedding = model.encode_image(preprocessed_image)

    vectors.append(
        {
            "embedding": image_embedding,
            "image_path": str(image_path)
        }
    )

In [22]:
text = clip.tokenize("big tree").to(device)

with torch.no_grad():
    text_features = model.encode_text(text)

In [21]:
vectors

[{'embedding': tensor([[ 2.3264e-01, -1.5028e-01,  1.3719e-01, -2.8007e-01, -6.8616e-02,
           -2.5566e-01,  1.2112e-01,  6.5322e-01,  2.5998e-01,  3.7328e-01,
            5.2039e-01, -7.6153e-01,  5.0031e-01,  5.0962e-02,  1.6783e-02,
           -2.5473e-01, -1.6868e-01, -1.8853e-01,  2.3283e-01, -1.7544e-01,
            5.2466e-02,  2.7805e-01, -4.8855e-02,  1.9429e-01, -6.9070e-01,
            1.3013e-01,  6.5368e-02,  1.9967e-01,  1.1857e-01, -8.0304e-02,
            9.2452e-02,  8.2358e-02, -2.8882e-02, -3.4347e-02,  1.4490e-01,
            5.8544e-01, -1.5305e-01, -3.5207e-01,  2.4370e-01, -3.9087e-01,
           -6.0280e-01, -6.1334e-01, -5.9388e-01,  2.6728e-01, -9.9881e-02,
            3.0951e+00,  1.5715e-01, -2.1816e-03,  3.2742e-01, -5.0982e-01,
           -1.5914e-01,  1.8570e-01,  2.9888e-01,  8.8261e-02,  1.2477e-01,
            3.0552e-02,  6.0625e-01,  3.6132e-01, -1.3623e-01, -4.3202e-01,
            8.6239e-01, -1.8371e-01, -2.2892e-01,  2.1093e-01, -3.3677e-01,

In [60]:
db_tensor = torch.stack([
    torch.as_tensor(d["embedding"]).squeeze().flatten()
    for d in vectors
])

query = text_features
query = query.view(1, -1)

db_tensor = F.normalize(db_tensor, dim=1)
query = F.normalize(query, dim=1)

scores = query @ db_tensor.T

values, indices = scores.topk(3, dim=1)
print(indices)

tensor([[3, 1, 4]])


In [2]:
text = clip.tokenize(["a cat", "a cat with emoji", "a cat with its paw and emoji overlay"]).to(device)

with torch.no_grad():
    image_features = model.encode_image(image)
    text_features = model.encode_text(text)

    logits_per_image, logits_per_text = model(image, text)
    probs = logits_per_image.softmax(dim=-1).cpu().numpy()

print("Label probs:", probs)

Label probs: [[0.11536077 0.5451609  0.3394783 ]]
